# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [2]:

# ============================================================
# IMPORTS
# ============================================================

import os
from dotenv import load_dotenv
from openai import OpenAI

import gradio as gr


# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================
# Load the API keys and other environment variables from
# the .env file.
#
# Using a .env file keeps sensitive API keys out of the
# Python source code.

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")


# ============================================================
# CHECK API KEYS
# ============================================================
# Check whether our API keys were successfully loaded.
#
# We only print the beginning of each key for debugging.
# NEVER print your complete API key.

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


# ============================================================
# CONNECT TO OPENAI
# ============================================================
# Create an OpenAI client.
#
# We can use the OpenAI Python library to communicate with
# OpenAI's API.

openai = OpenAI()


# ============================================================
# CONNECT TO GOOGLE GEMINI
# ============================================================
# Google provides an OpenAI-compatible API endpoint.
#
# Because Gemini supports this interface, we can use the
# OpenAI Python client while changing the base URL to
# Google's Gemini endpoint.

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"


# Create the Gemini client using our Google API key.

gemini = OpenAI(
    api_key=google_api_key,
    base_url=gemini_url
)


# ============================================================
# GEMINI MODEL
# ============================================================
# Store the model name in a variable.
#
# This makes it easier to change models later without
# having to search through the entire program.

MODEL_GEMINI = "gemini-3-flash-preview"


# ============================================================
# SYSTEM MESSAGE
# ============================================================
# The system message controls how Gemini should behave
# when responding to the user.

system_message = "You are a helpful assistant."


# ============================================================
# GEMINI MESSAGE FUNCTION
# ============================================================
# This function takes a user's prompt and sends it to Gemini.
#
# We create two messages:
#
# 1. system - tells Gemini how it should behave
# 2. user   - contains the user's actual question
#
# The function then returns Gemini's response as text.

def message_gemini(prompt):

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Send the messages to the Gemini API.

    response = gemini.chat.completions.create(
        model=MODEL_GEMINI,
        messages=messages
    )

    # Return only the text generated by Gemini.

    return response.choices[0].message.content


# ============================================================
# SIMPLE FUNCTION EXAMPLE
# ============================================================
# This was an earlier example used to learn how functions
# work with Gradio.
#
# It takes text and converts it to uppercase.

def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()


# Test the shout function.

print(shout("hello"))


# ============================================================
# BASIC GRADIO INTERFACE
# ============================================================
# This is an earlier Gradio example.
#
# It has been commented out because we are now using Gemini
# instead of the simple shout function.
#
# The auth parameter can be used to add username/password
# authentication to a Gradio application.

# gr.Interface(
#     fn=shout,
#     inputs="textbox",
#     outputs="textbox",
#     flagging_mode="never"
# ).launch(
#     inbrowser=True,
#     auth=("aidan", "aidan13@")
# )


# ============================================================
# FORCE DARK MODE IN GRADIO
# ============================================================
# JavaScript can be passed to Gradio to customize the
# behaviour of the web interface.
#
# This JavaScript checks the URL for the dark theme and
# redirects the page to the dark theme if necessary.

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);

    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""


# Earlier dark-mode Gradio example.
# Commented out because we are using the newer interface below.

# gr.Interface(
#     fn=shout,
#     inputs="textbox",
#     outputs="textbox",
#     flagging_mode="never",
#     js=force_dark_mode
# ).launch()


# ============================================================
# EARLIER GEMINI GRADIO INTERFACE
# ============================================================
# This was an earlier version of the Gemini interface.
#
# It is kept here so we can see the progression of the
# application as we learn more about Gradio.

# message_input = gr.Textbox(
#     label="Your message",
#     info="Ask Gemini anything",
#     lines=7
# )

# message_output = gr.Textbox(
#     label="Gemini Response",
#     lines=8
# )


# view = gr.Interface(
#     fn=message_gemini,
#     title="Gemini Assistant",
#     inputs=[message_input],
#     outputs=[message_output],
#     examples=[
#         "Explain APIs to me",
#         "What is Python?",
#         "Explain machine learning simply"
#     ],
#     flagging_mode="never"
# )

# view.launch()


# ============================================================
# MARKDOWN GEMINI INTERFACE
# ============================================================
# We now tell Gemini to respond using Markdown.
#
# Markdown allows the response to be displayed with
# formatting such as:
#
# - Headings
# - Bullet points
# - Bold text
# - Lists
# - Other Markdown formatting

system_message = (
    "You are a helpful assistant that responds in markdown "
    "without code blocks"
)


message_input = gr.Textbox(
    label="Your message:",
    info="Enter a message for Gemini",
    lines=7
)


message_output = gr.Markdown(
    label="Response"
)


view = gr.Interface(
    fn=message_gemini,
    title="GEMINI",
    inputs=message_input,
    outputs=message_output,
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI Engineer",
    ],
    flagging_mode="never"
)


# ============================================================
# STREAMING GEMINI RESPONSES
# ============================================================
# Instead of waiting for Gemini to generate the entire
# response before displaying anything, streaming allows
# us to receive the response piece by piece.
#
# The `yield` keyword allows this function to return
# partial results as Gemini generates them.


def stream_gemini(prompt):

    # Create the messages that will be sent to Gemini.

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # stream=True tells Gemini to send the response
    # in chunks instead of waiting for the full response.

    stream = gemini.chat.completions.create(
        model=MODEL_GEMINI,
        messages=messages,
        stream=True
    )

   

    result = ""

    

    for chunk in stream:

        # Add the new piece of text to our existing result.
        #
        # Some chunks may not contain any text, which is
        # why we use `or ""`.

        result += chunk.choices[0].delta.content or ""

        # yield sends the current result back to the caller
        # without ending the function.
       

        yield result


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ.Ab8RN
Shout has been called with input hello
HELLO
